# Téléchargement & exploration

possible run en local (aucun gpu require)

Dataset réel : [`b-mc2/sql-create-context`](https://huggingface.co/datasets/b-mc2/sql-create-context) (question + schéma `CREATE TABLE` → requête SQL). Ce notebook prototype avant d'industrialiser dans `src/download_dataset.py`.

import sys
sys.path.append('../src')
import json
import pandas as pd
from datasets import load_dataset
from prompts import build_instruction

## télécharge les données brute & exploit

ds = load_dataset('b-mc2/sql-create-context', split='train')
print(len(ds), ds.column_names)
pd.DataFrame(ds[:5])

## Longueurs (pour choisir `max_seq_length`)

Le prompt contient le schéma : vérifier que 512 tokens suffisent (≈ 4 caractères par token).

In [ ]:
df = pd.DataFrame(ds.shuffle(seed=42).select(range(2000)))
df['n_chars'] = (df['context'] + df['question'] + df['answer']).str.len()
print(df['n_chars'].describe())
print('Part > 1800 caractères (~450 tokens):', (df['n_chars'] > 1800).mean().round(3))

## Générer train/val avec le script

Sous-échantillon reproductible (seed 42) : 2000 train / 200 val.

In [ ]:
train = [json.loads(l) for l in open('../data/processed/train.jsonl', encoding='utf-8')]
print(train[0]['instruction'])
print('\n--> ', train[0]['output'])